# Image Classification System for Object Recognition
## Results and split verification
This notebook audits the submitted artifacts without downloading image data or retraining.
The full training and evaluation entry points are documented in the README.
Run from the repository root or notebooks folder. AI assistance with code and documentation is disclosed in the README.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np

root = Path.cwd()
if not (root / "results").is_dir():
    root = root.parent
assert (root / "src/dataset.py").is_file()
manifest = json.loads((root / "results/split_manifest.json").read_text())
archive = root / "results" / manifest["archive"]
assert hashlib.sha256(archive.read_bytes()).hexdigest() == manifest["sha256"]
print("Split archive checksum verified.")

Split archive checksum verified.


## Exact data membership
All arrays use original dataset indices. Training and validation index the official 50,000-image training dataset. Test indices refer to a separate 10,000-image dataset.

In [2]:
with np.load(archive, allow_pickle=False) as arrays:
    splits = {key: arrays[key] for key in arrays.files}
expected = {"train":40000, "validation":10000, "test":10000, "svm_fit":32000, "svm_selection":8000}
for name, count in expected.items():
    assert len(splits[name]) == count
    assert len(np.unique(splits[name])) == count
assert not np.intersect1d(splits["train"], splits["validation"]).size
assert np.array_equal(np.sort(np.concatenate([splits["train"], splits["validation"]])), np.arange(50000))
assert not np.intersect1d(splits["svm_fit"], splits["svm_selection"]).size
assert np.array_equal(np.sort(np.concatenate([splits["svm_fit"], splits["svm_selection"]])), np.sort(splits["train"]))
assert np.array_equal(splits["test"], np.arange(10000))
print("Verified sizes, uniqueness, disjoint partitions, and complete dataset coverage.")
print(expected)

Verified sizes, uniqueness, disjoint partitions, and complete dataset coverage.
{'train': 40000, 'validation': 10000, 'test': 10000, 'svm_fit': 32000, 'svm_selection': 8000}


## Original benchmark results
Macro-F1 weights classes equally. Confusion counts let us independently recompute both published headline scores and identify the most frequent class confusion.

In [3]:
for name in ("hog_svm", "mobilenet_v2"):
    report = json.loads((root / f"results/{name}_test_metrics.json").read_text())
    counts = np.asarray(report["confusion_matrix"])
    assert counts.shape == (10, 10) and counts.sum() == 10000
    assert np.array_equal(counts.sum(axis=1), np.full(10, 1000))
    correct = counts.diagonal()
    accuracy = float(correct.sum() / counts.sum())
    macro_f1 = float((2 * correct / (counts.sum(axis=0) + counts.sum(axis=1))).mean())
    assert np.isclose(accuracy, report["accuracy"])
    assert np.isclose(macro_f1, report["macro_f1"])
    print(f"{name}: accuracy={accuracy:.2%}, macro-F1={macro_f1:.4f}, cat-to-dog errors={counts[3,5]}")

hog_svm: accuracy=54.77%, macro-F1=0.5415, cat-to-dog errors=170
mobilenet_v2: accuracy=91.48%, macro-F1=0.9148, cat-to-dog errors=83


## Additional training experiment
The selected checkpoint must maximize validation macro-F1. Test scores describe the selected recipes after that decision. CIFAR-100 transfer adds both auxiliary data and training time, so its effect is not an isolated data-only comparison.

In [4]:
refinement = json.loads((root / "results/mobilenet_v2_refinement.json").read_text())
selected = max(refinement["runs"], key=lambda name: refinement["runs"][name]["validation"]["macro_f1"])
assert selected == refinement["selected"]
assert refinement["cifar100_audit"]["used"] == 49999
assert refinement["cifar10_1_samples"] == 2000
for name, run in refinement["runs"].items():
    print(f'{name}: validation F1={run["validation"]["macro_f1"]:.4f}, CIFAR-10 accuracy={run["cifar10_test"]["accuracy"]:.2%}, CIFAR-10.1 accuracy={run["cifar10_1_v6"]["accuracy"]:.2%}')
print("Validation-selected checkpoint:", selected)

original: validation F1=0.9170, CIFAR-10 accuracy=91.48%, CIFAR-10.1 accuracy=82.40%
cifar10_only: validation F1=0.9478, CIFAR-10 accuracy=94.28%, CIFAR-10.1 accuracy=86.45%
cifar100_transfer: validation F1=0.9480, CIFAR-10 accuracy=94.42%, CIFAR-10.1 accuracy=88.05%
Validation-selected checkpoint: cifar100_transfer


## Interpretation
Most of the original-benchmark improvement comes from the stronger CIFAR-10 recipe.
Auxiliary training adds 0.14 percentage points of accuracy on CIFAR-10 and 1.60 on CIFAR-10.1.
One seed limits claims about repeatability. Test-score exposure during development is disclosed on the website.
High softmax probability does not establish correctness on an unfamiliar upload.
The original and selected refined weights are included. Intermediate candidate weights were not retained.
## Reproduce model outputs
Use the README commands for data access, training, evaluation, and browser export.
The original test reports include per-class precision/recall, confusion counts, and error indices.
The refinement JSON includes every recorded epoch and aggregate scores for the three runs.